# AeroFootprint — WHU Building Extraction (TensorFlow U-Net)

**Training from scratch. No pretrained checkpoint. No AWS account required.**

This notebook uses the public `giswqs/WHU-Building-Dataset` Hugging Face dataset mirror. The dataset card specifies 512×512 RGB PNG tiles at 0.3 m resolution, binary masks (`0` background, `255` building), and official train/validation/test splits of 5,732 / 1,228 / 1,228 tiles. Its reported repository size is 5.25 GB and the card declares CC BY 4.0. See the linked dataset card and cite the original WHU paper in any public report.

**Important scope note:** this Hugging Face mirror contains ordinary PNG tiles and pixel masks, not per-tile GeoTIFF affine transforms. This notebook can produce building segmentation masks and polygon contours in **image-pixel coordinates**. It cannot truthfully produce latitude/longitude georeferenced footprints from these files alone. A later georeferenced inference stage needs imagery with CRS/affine metadata.

## Run order

1. Select a GPU runtime (`Runtime → Change runtime type → T4 GPU` or another available GPU).
2. Run every cell from top to bottom.
3. Authorize Google Drive when prompted. No AWS key, AWS account, or Hugging Face token is required for this public dataset.
4. The full dataset download is about 5.25 GB. It is downloaded to Colab's temporary disk; checkpoints and results are saved to Google Drive.

If Colab disconnects, reconnect and rerun cells from the beginning using the **same `RUN_ID`** to recover from the epoch backup. Change `RUN_ID` to start a separate experiment.

### Sources

- Dataset mirror and card: https://huggingface.co/datasets/giswqs/WHU-Building-Dataset
- Original WHU Building Dataset page: https://gpcv.whu.edu.cn/data/building_dataset.html
- Paper: Ji, S., Wei, S., & Lu, M. (2019). *Fully convolutional networks for multisource building identification*. IEEE Transactions on Geoscience and Remote Sensing, 57(1), 108–120.

## 1. Select and verify the Colab runtime

In [ ]:
import os, sys, platform, shutil, subprocess, json, math, random, time
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

# Colab normally has TensorFlow preinstalled. Do not replace its TensorFlow build blindly.
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if not gpus:
    raise RuntimeError(
        "No GPU is attached. In Colab select Runtime > Change runtime type > T4 GPU "
        "(or another available GPU), reconnect, and rerun this cell. Full training on CPU is not recommended."
    )
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Could not set memory growth:", exc)


## 2. Install dataset-download and image utilities

In [ ]:
%pip -q install "huggingface_hub[hf_xet]" pillow opencv-python-headless matplotlib pandas tqdm

## 3. Mount Google Drive and set experiment paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset files are kept on the temporary Colab disk for faster I/O.
DATA_ROOT = Path('/content/whu-building-dataset')

# Models, checkpoints, plots, metrics and predictions persist in Google Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/AeroFootprint')
RUN_ID = 'whu_unet_v1'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
MODEL_DIR = RUN_DIR / 'models'
ARTIFACT_DIR = RUN_DIR / 'artifacts'
OUTPUT_DIR = RUN_DIR / 'outputs'
BACKUP_DIR = RUN_DIR / 'backup'

for folder in [DRIVE_ROOT, RUN_DIR, MODEL_DIR, ARTIFACT_DIR, OUTPUT_DIR, BACKUP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Run directory:', RUN_DIR)
print('Free Colab disk (GB):', round(shutil.disk_usage('/content').free / (1024**3), 2))

## 4. Configure the WHU dataset and training run

In [ ]:
DATASET_REPO = 'giswqs/WHU-Building-Dataset'
DATASET_URL = 'https://huggingface.co/datasets/giswqs/WHU-Building-Dataset'
LICENSE = 'CC BY 4.0 (as declared by the Hugging Face dataset card)'

IMAGE_SIZE = 256              # random 256x256 training crops from 512x512 tiles
TILE_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 25
BASE_FILTERS = 32
LEARNING_RATE = 1e-4
SEED = 42
MIN_FREE_GB = 15

EXPECTED_COUNTS = {'train': 5732, 'val': 1228, 'test': 1228}

random.seed(SEED)
np_seed_set = False
try:
    import numpy as np
    np.random.seed(SEED)
    np_seed_set = True
except Exception:
    pass
tf.keras.utils.set_random_seed(SEED)

free_gb = shutil.disk_usage('/content').free / (1024**3)
if free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f'Only {free_gb:.1f} GB free on /content. At least {MIN_FREE_GB} GB is recommended '
        'for the 5.25 GB dataset plus temporary download/cache space. Switch to a runtime with more disk.'
    )

print('Dataset:', DATASET_REPO)
print('Expected image counts:', EXPECTED_COUNTS)
print('GPU batch size:', BATCH_SIZE)
print('Epoch target:', EPOCHS)

## 5. Download the public dataset from Hugging Face (no AWS)

In [ ]:
from huggingface_hub import snapshot_download

# Public repository: no token/login is required. Download only the six image/mask folders.
allow_patterns = [
    'train/Image/*.png', 'train/Mask/*.png',
    'val/Image/*.png',   'val/Mask/*.png',
    'test/Image/*.png',  'test/Mask/*.png',
    'README.md'
]

local_ready = True
for split, expected in EXPECTED_COUNTS.items():
    for subfolder in ['Image', 'Mask']:
        folder = DATA_ROOT / split / subfolder
        count = len(list(folder.glob('*.png'))) if folder.is_dir() else 0
        if count != expected:
            local_ready = False
            print(f'{split}/{subfolder}: found {count}; expected {expected}. Download/resume required.')

if local_ready:
    print('All dataset folders already exist with expected counts; reusing local download.')
else:
    print('Downloading/resuming the public WHU dataset. Repository size is reported as approximately 5.25 GB.')
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        local_dir=str(DATA_ROOT),
        allow_patterns=allow_patterns,
        max_workers=8,
        token=None
    )

print('Dataset root:', DATA_ROOT)
for split in ['train', 'val', 'test']:
    image_dir = DATA_ROOT / split / 'Image'
    mask_dir = DATA_ROOT / split / 'Mask'
    print(split, '| images:', len(list(image_dir.glob('*.png'))), '| masks:', len(list(mask_dir.glob('*.png'))))


## 6. Match image/mask pairs and validate the dataset

In [ ]:
from PIL import Image
import numpy as np
import pandas as pd


def build_pairs(split):
    image_dir = DATA_ROOT / split / 'Image'
    mask_dir = DATA_ROOT / split / 'Mask'
    if not image_dir.is_dir() or not mask_dir.is_dir():
        raise FileNotFoundError(f'Missing Image/Mask folders for split={split}: {image_dir}, {mask_dir}')

    image_files = sorted(image_dir.glob('*.png'))
    mask_files = sorted(mask_dir.glob('*.png'))
    image_map = {p.stem: p for p in image_files}
    mask_map = {p.stem: p for p in mask_files}

    missing_masks = sorted(set(image_map) - set(mask_map))
    missing_images = sorted(set(mask_map) - set(image_map))
    if missing_masks or missing_images:
        raise RuntimeError(
            f'Pair mismatch in {split}: missing masks={missing_masks[:5]}, '
            f'missing images={missing_images[:5]}'
        )
    pairs = [(image_map[k], mask_map[k]) for k in sorted(image_map)]
    if not pairs:
        raise RuntimeError(f'No image/mask pairs found in {split}')
    return pairs


train_pairs = build_pairs('train')
val_pairs = build_pairs('val')
test_pairs = build_pairs('test')

actual_counts = {'train': len(train_pairs), 'val': len(val_pairs), 'test': len(test_pairs)}
print('Actual paired counts:', actual_counts)
for split, expected in EXPECTED_COUNTS.items():
    if actual_counts[split] != expected:
        raise RuntimeError(f'{split}: expected {expected} pairs per dataset card, found {actual_counts[split]}. Stop and inspect dataset version/layout.')

# Validate a sample from each split: file sizes, channels, binary mask values.
for split, pairs in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
    image_path, mask_path = pairs[0]
    with Image.open(image_path) as im:
        image_arr = np.asarray(im.convert('RGB'))
    with Image.open(mask_path) as ma:
        mask_arr = np.asarray(ma.convert('L'))
    print(split, image_path.name, image_arr.shape, image_arr.dtype,
          '| mask:', mask_arr.shape, mask_arr.dtype,
          '| mask values sample:', np.unique(mask_arr)[:10])
    if image_arr.shape != (TILE_SIZE, TILE_SIZE, 3):
        raise ValueError(f'Unexpected image shape: {image_arr.shape}')
    if mask_arr.shape != (TILE_SIZE, TILE_SIZE):
        raise ValueError(f'Unexpected mask shape: {mask_arr.shape}')

print('All image/mask pairs match and sample shapes are valid.')

## 7. Visualize raw images and masks before training

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, (split, pairs) in enumerate([('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]):
    img_path, mask_path = pairs[min(10, len(pairs)-1)]
    image = np.asarray(Image.open(img_path).convert('RGB'))
    mask = np.asarray(Image.open(mask_path).convert('L')) > 0
    axes[0, col].imshow(image)
    axes[0, col].set_title(f'{split}: {img_path.stem}')
    axes[1, col].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'{split}: binary building mask')
    axes[0, col].axis('off'); axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## 8. Build TensorFlow input pipelines (finite-epoch bug fixed)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_pair(image_path, mask_path):
    image_bytes = tf.io.read_file(image_path)
    mask_bytes = tf.io.read_file(mask_path)
    image = tf.io.decode_png(image_bytes, channels=3)
    mask = tf.io.decode_png(mask_bytes, channels=1)
    image = tf.image.convert_image_dtype(image, tf.float32)  # uint8 [0,255] -> float32 [0,1]
    mask = tf.cast(mask > 0, tf.float32)                    # label 255 -> 1
    image = tf.ensure_shape(image, [TILE_SIZE, TILE_SIZE, 3])
    mask = tf.ensure_shape(mask, [TILE_SIZE, TILE_SIZE, 1])
    return image, mask


def decode_random_train_crop(image_path, mask_path):
    image, mask = decode_pair(image_path, mask_path)
    joined = tf.concat([image, mask], axis=-1)
    joined = tf.image.random_crop(joined, size=[IMAGE_SIZE, IMAGE_SIZE, 4])
    image, mask = joined[..., :3], joined[..., 3:]

    # Geometry transforms are applied jointly to image and mask.
    if tf.random.uniform(()) > 0.5:
        image, mask = tf.image.flip_left_right(image), tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image, mask = tf.image.flip_up_down(image), tf.image.flip_up_down(mask)
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    image, mask = tf.image.rot90(image, k), tf.image.rot90(mask, k)

    # Radiometric augmentation changes imagery only.
    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.90, upper=1.10)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask


def four_validation_crops(image, mask):
    image_patches = tf.stack([
        image[0:256, 0:256, :], image[0:256, 256:512, :],
        image[256:512, 0:256, :], image[256:512, 256:512, :]
    ], axis=0)
    mask_patches = tf.stack([
        mask[0:256, 0:256, :], mask[0:256, 256:512, :],
        mask[256:512, 0:256, :], mask[256:512, 256:512, :]
    ], axis=0)
    return tf.data.Dataset.from_tensor_slices((image_patches, mask_patches))


def make_eval_dataset(pairs, batch_size=16):
    image_paths = [str(p[0]) for p in pairs]
    mask_paths = [str(p[1]) for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    ds = ds.map(decode_pair, num_parallel_calls=AUTOTUNE)
    ds = ds.flat_map(four_validation_crops)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


train_image_paths = [str(p[0]) for p in train_pairs]
train_mask_paths = [str(p[1]) for p in train_pairs]
train_ds = tf.data.Dataset.from_tensor_slices((train_image_paths, train_mask_paths))
train_ds = train_ds.shuffle(min(len(train_pairs), 2048), seed=SEED, reshuffle_each_iteration=True)
train_ds = train_ds.map(decode_random_train_crop, num_parallel_calls=AUTOTUNE, deterministic=False)
# Important: batch first, then repeat. This avoids the earlier OUT_OF_RANGE / zero-loss epoch bug.
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=False).repeat().prefetch(AUTOTUNE)

val_ds = make_eval_dataset(val_pairs, batch_size=16)  # finite: validation exhausts normally
val_patches_per_epoch = len(val_pairs) * 4
steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SIZE)
validation_steps = math.ceil(val_patches_per_epoch / 16)

print('Train tiles:', len(train_pairs))
print('Training steps per epoch:', steps_per_epoch)
print('Validation patches:', val_patches_per_epoch)
print('Validation batches:', validation_steps)

## 9. Smoke-check one training and validation batch

In [ ]:
for xb, yb in train_ds.take(1):
    print('Train batch:', xb.shape, yb.shape,
          '| image range:', float(tf.reduce_min(xb)), float(tf.reduce_max(xb)),
          '| mask values:', np.unique(yb.numpy()))
for xb, yb in val_ds.take(1):
    print('Validation batch:', xb.shape, yb.shape,
          '| mask values:', np.unique(yb.numpy()))

## 10. Define our U-Net, loss and metrics

In [ ]:
from tensorflow.keras import layers


@tf.keras.utils.register_keras_serializable(package='AeroFootprint')
def bce_dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    bce_map = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce_map, axis=[1, 2])
    axes = [1, 2, 3]
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred, axis=axes)
    dice = (2.0 * intersection + 1.0) / (denominator + 1.0)
    return bce + (1.0 - dice)


@tf.keras.utils.register_keras_serializable(package='AeroFootprint')
class OverlapMetric(tf.keras.metrics.Metric):
    def __init__(self, kind='dice', threshold=0.5, name=None, **kwargs):
        super().__init__(name=name or kind, **kwargs)
        self.kind = kind
        self.threshold = threshold
        self.tp = self.add_weight(name='tp', initializer='zeros')
        self.fp = self.add_weight(name='fp', initializer='zeros')
        self.fn = self.add_weight(name='fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true >= 0.5, tf.float32)
        yp = tf.cast(y_pred >= self.threshold, tf.float32)
        self.tp.assign_add(tf.reduce_sum(yt * yp))
        self.fp.assign_add(tf.reduce_sum((1.0 - yt) * yp))
        self.fn.assign_add(tf.reduce_sum(yt * (1.0 - yp)))

    def result(self):
        eps = tf.keras.backend.epsilon()
        if self.kind == 'dice':
            return (2.0 * self.tp + eps) / (2.0 * self.tp + self.fp + self.fn + eps)
        return (self.tp + eps) / (self.tp + self.fp + self.fn + eps)

    def reset_state(self):
        for variable in [self.tp, self.fp, self.fn]:
            variable.assign(0.0)

    def get_config(self):
        config = super().get_config()
        config.update({'kind': self.kind, 'threshold': self.threshold})
        return config


def conv_block(x, filters, dropout_rate=0.0):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout_rate:
        x = layers.Dropout(dropout_rate)(x)
    return x


def build_unet(input_shape=(256, 256, 3), base_filters=32):
    inputs = layers.Input(shape=input_shape, name='rgb_tile')
    skips = []
    x = inputs
    for level in range(4):
        filters = base_filters * (2 ** level)
        x = conv_block(x, filters)
        skips.append(x)
        x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = conv_block(x, base_filters * 16, dropout_rate=0.30)
    for level in reversed(range(4)):
        filters = base_filters * (2 ** level)
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
        x = layers.Concatenate()([x, skips[level]])
        x = conv_block(x, filters)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='building_probability')(x)
    return tf.keras.Model(inputs, outputs, name='AeroFootprint_WHUNet')


# T4/L4 GPUs support mixed precision; keep output/loss computation in float32.
if tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision policy enabled for GPU.')
else:
    tf.keras.mixed_precision.set_global_policy('float32')
    print('Using float32 CPU policy.')

tf.keras.backend.clear_session()
model = build_unet(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), base_filters=BASE_FILTERS)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=bce_dice_loss,
    metrics=[
        OverlapMetric(kind='dice', name='dice'),
        OverlapMetric(kind='iou', name='iou'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)
model.summary()

## 11. Train from random initialization and save checkpoints to Drive

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / 'best_model.keras'
FINAL_MODEL_PATH = MODEL_DIR / 'final_model.keras'
HISTORY_PATH = ARTIFACT_DIR / 'training_history.csv'

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH), monitor='val_dice', mode='max',
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice', mode='max', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice', mode='max', patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(HISTORY_PATH), append=HISTORY_PATH.exists()),
    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR), save_freq='epoch', delete_checkpoint=False
    )
]

print('Random initialization: yes')
print('Pretrained weights loaded: no')
print('Train steps/epoch:', steps_per_epoch)
print('Validation steps:', validation_steps)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

model.save(FINAL_MODEL_PATH)
print('Best checkpoint:', BEST_MODEL_PATH)
print('Final model:', FINAL_MODEL_PATH)

## 12. Plot the training history

In [ ]:
history_df = pd.read_csv(HISTORY_PATH)
print(history_df.tail().to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if 'loss' in history_df:
    axes[0].plot(history_df['loss'], label='train loss')
if 'val_loss' in history_df:
    axes[0].plot(history_df['val_loss'], label='validation loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)
if 'dice' in history_df:
    axes[1].plot(history_df['dice'], label='train Dice')
if 'val_dice' in history_df:
    axes[1].plot(history_df['val_dice'], label='validation Dice')
axes[1].set_title('Dice'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 13. Load the best checkpoint and tune threshold on validation only

In [ ]:
# Loading compile=False avoids requiring optimizer restoration for inference.
best_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
thresholds = np.round(np.arange(0.25, 0.751, 0.025), 3)
counts = {float(t): {'tp': 0, 'fp': 0, 'fn': 0} for t in thresholds}

for xb, yb in val_ds:
    probs = best_model(xb, training=False).numpy()
    gt = yb.numpy() >= 0.5
    for t in thresholds:
        pred = probs >= t
        c = counts[float(t)]
        c['tp'] += int(np.logical_and(gt, pred).sum())
        c['fp'] += int(np.logical_and(~gt, pred).sum())
        c['fn'] += int(np.logical_and(gt, ~pred).sum())

val_rows = []
for t in thresholds:
    c = counts[float(t)]
    dice = (2*c['tp'] + 1e-7) / (2*c['tp'] + c['fp'] + c['fn'] + 1e-7)
    iou = (c['tp'] + 1e-7) / (c['tp'] + c['fp'] + c['fn'] + 1e-7)
    precision = (c['tp'] + 1e-7) / (c['tp'] + c['fp'] + 1e-7)
    recall = (c['tp'] + 1e-7) / (c['tp'] + c['fn'] + 1e-7)
    val_rows.append({'threshold': float(t), 'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall})
val_threshold_df = pd.DataFrame(val_rows)
# Tie-break toward 0.5 for a less extreme operating point.
val_threshold_df['distance_from_0_5'] = (val_threshold_df['threshold'] - 0.5).abs()
val_threshold_df = val_threshold_df.sort_values(['dice', 'distance_from_0_5'], ascending=[False, True])
BEST_THRESHOLD = float(val_threshold_df.iloc[0]['threshold'])
val_threshold_df.to_csv(ARTIFACT_DIR / 'validation_threshold_sweep.csv', index=False)

with open(ARTIFACT_DIR / 'threshold.json', 'w') as f:
    json.dump({'selected_threshold': BEST_THRESHOLD, 'selection_split': 'validation',
               'selection_metric': 'micro_dice', 'threshold_candidates': thresholds.tolist()}, f, indent=2)

print('Selected threshold (validation only):', BEST_THRESHOLD)
print(val_threshold_df.head(10).to_string(index=False))

## 14. Evaluate once on the untouched official test split

In [ ]:
import cv2

TEST_PRED_DIR = OUTPUT_DIR / 'test_predictions'
TEST_VIS_DIR = OUTPUT_DIR / 'test_visualizations'
TEST_POLYGON_DIR = OUTPUT_DIR / 'pixel_coordinate_footprints'
for folder in [TEST_PRED_DIR, TEST_VIS_DIR, TEST_POLYGON_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

test_ds = make_eval_dataset(test_pairs, batch_size=16)


def stitch_four(arr4):
    # arr4 order: top-left, top-right, bottom-left, bottom-right
    top = np.concatenate([arr4[0], arr4[1]], axis=1)
    bottom = np.concatenate([arr4[2], arr4[3]], axis=1)
    return np.concatenate([top, bottom], axis=0)


def scores_from_counts(tp, fp, fn):
    eps = 1e-7
    return {
        'dice': float((2*tp + eps) / (2*tp + fp + fn + eps)),
        'iou': float((tp + eps) / (tp + fp + fn + eps)),
        'precision': float((tp + eps) / (tp + fp + eps)),
        'recall': float((tp + eps) / (tp + fn + eps))
    }


def save_pixel_geojson(mask, destination, image_id, min_area=12):
    # This is deliberately pixel-coordinate GeoJSON, not a geographic CRS.
    contours, _ = cv2.findContours((mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    features = []
    for contour in contours:
        area = float(cv2.contourArea(contour))
        if area < min_area:
            continue
        epsilon = max(1.0, 0.002 * cv2.arcLength(contour, True))
        polygon = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
        if len(polygon) < 3:
            continue
        coords = [[int(x), int(y)] for x, y in polygon.tolist()]
        coords.append(coords[0])
        features.append({
            'type': 'Feature',
            'properties': {'image_id': image_id, 'area_pixels': area, 'coordinate_space': 'pixel'},
            'geometry': {'type': 'Polygon', 'coordinates': [coords]}
        })
    collection = {
        'type': 'FeatureCollection',
        'name': f'{image_id}_pixel_footprints',
        'coordinate_reference': 'image pixel coordinates (origin at upper-left; x=column, y=row); not georeferenced',
        'features': features
    }
    with open(destination, 'w') as f:
        json.dump(collection, f)


per_image_rows = []
total_tp = total_fp = total_fn = 0
sample_visuals = 12
image_index = 0

for xb, yb in test_ds:
    probs_batch = best_model(xb, training=False).numpy()[..., 0]
    masks_batch = yb.numpy()[..., 0] >= 0.5
    if len(probs_batch) % 4 != 0:
        raise RuntimeError('Test patch batch is not divisible by four; cannot reconstruct 512x512 tiles safely.')

    for offset in range(0, len(probs_batch), 4):
        prob_full = stitch_four(probs_batch[offset:offset+4])
        gt_full = stitch_four(masks_batch[offset:offset+4].astype(np.uint8)).astype(bool)
        pred_full = prob_full >= BEST_THRESHOLD
        tp = int(np.logical_and(gt_full, pred_full).sum())
        fp = int(np.logical_and(~gt_full, pred_full).sum())
        fn = int(np.logical_and(gt_full, ~pred_full).sum())
        total_tp += tp; total_fp += fp; total_fn += fn

        image_path, mask_path = test_pairs[image_index]
        image_id = image_path.stem
        metrics = scores_from_counts(tp, fp, fn)
        per_image_rows.append({'image_id': image_id, 'tp': tp, 'fp': fp, 'fn': fn,
                               'building_pixels_gt': int(gt_full.sum()),
                               'building_pixels_pred': int(pred_full.sum()), **metrics})

        Image.fromarray((pred_full.astype(np.uint8) * 255)).save(TEST_PRED_DIR / f'{image_id}_mask.png')

        if image_index < sample_visuals:
            rgb = np.asarray(Image.open(image_path).convert('RGB'))
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(rgb); axes[0].set_title('RGB image')
            axes[1].imshow(gt_full, cmap='gray'); axes[1].set_title('Ground truth')
            axes[2].imshow(pred_full, cmap='gray'); axes[2].set_title('Prediction')
            overlay = rgb.copy()
            # Red = false positives; green = true positives; blue = false negatives.
            overlay[np.logical_and(pred_full, gt_full)] = [0, 220, 0]
            overlay[np.logical_and(pred_full, ~gt_full)] = [255, 0, 0]
            overlay[np.logical_and(~pred_full, gt_full)] = [0, 80, 255]
            axes[3].imshow(overlay); axes[3].set_title('Overlay: TP green / FP red / FN blue')
            for ax in axes: ax.axis('off')
            plt.tight_layout()
            fig.savefig(TEST_VIS_DIR / f'{image_id}_comparison.png', dpi=150, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            save_pixel_geojson(pred_full, TEST_POLYGON_DIR / f'{image_id}_pixel_footprints.geojson', image_id)

        image_index += 1

if image_index != len(test_pairs):
    raise RuntimeError(f'Processed {image_index} test images but expected {len(test_pairs)}.')

per_image_df = pd.DataFrame(per_image_rows)
per_image_df.to_csv(OUTPUT_DIR / 'metrics_per_image.csv', index=False)
micro = scores_from_counts(total_tp, total_fp, total_fn)
macro = {name: float(per_image_df[name].mean()) for name in ['dice', 'iou', 'precision', 'recall']}
summary = {
    'project': 'AeroFootprint', 'dataset': DATASET_REPO, 'license': LICENSE,
    'training': 'TensorFlow U-Net trained from random initialization; no pretrained weights',
    'split_policy': 'published WHU train/validation/test splits',
    'train_images': len(train_pairs), 'validation_images': len(val_pairs), 'test_images': len(test_pairs),
    'input_tile_size': [512, 512], 'model_patch_size': IMAGE_SIZE,
    'selected_threshold': BEST_THRESHOLD, 'threshold_selected_on': 'validation split only',
    'test_metrics_micro': micro, 'test_metrics_macro_per_image': macro,
    'test_pixel_counts': {'tp': total_tp, 'fp': total_fp, 'fn': total_fn},
    'geospatial_warning': 'The PNG mirror has no tile geotransform. Exported GeoJSON contours use pixel coordinates and are not latitude/longitude footprints.'
}
with open(OUTPUT_DIR / 'evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('TEST MICRO METRICS:', json.dumps(micro, indent=2))
print('TEST MACRO METRICS:', json.dumps(macro, indent=2))
print('Metrics saved to:', OUTPUT_DIR)
print(per_image_df.head().to_string(index=False))

## 15. Save run manifest, attribution and output inventory

In [ ]:
manifest = {
    'project': 'AeroFootprint',
    'model': 'U-Net (TensorFlow/Keras)',
    'initialization': 'random',
    'pretrained_checkpoint_used': False,
    'dataset_repo': DATASET_REPO,
    'dataset_url': DATASET_URL,
    'dataset_license_declared_by_mirror': LICENSE,
    'dataset_citation': 'Ji, S., Wei, S., & Lu, M. (2019). Fully convolutional networks for multisource building identification. IEEE TGRS, 57(1), 108–120.',
    'counts': actual_counts,
    'image_shape': [512, 512, 3],
    'mask_values': {'background': 0, 'building': 255},
    'training_patch_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'requested_epochs': EPOCHS,
    'base_filters': BASE_FILTERS,
    'learning_rate': LEARNING_RATE,
    'selected_threshold': BEST_THRESHOLD,
    'threshold_selection_split': 'validation',
    'tensorflow_version': tf.__version__,
    'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
    'important_limit': 'Dataset mirror has PNGs without georeferencing. Pixel-coordinate polygon exports are not geographic GeoJSON.'
}
with open(ARTIFACT_DIR / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

readme_text = (
    f"# AeroFootprint WHU run\n\nDataset: {DATASET_URL}\n"
    f"License declared by dataset mirror: {LICENSE}\n\n"
    "Citation: Ji, S., Wei, S., & Lu, M. (2019). Fully convolutional networks for multisource building identification. IEEE Transactions on Geoscience and Remote Sensing, 57(1), 108–120.\n\n"
    "Model: TensorFlow U-Net trained from random initialization; no pretrained checkpoint.\n\n"
    f"Selected threshold: {BEST_THRESHOLD} (selected using validation split only).\n\n"
    "IMPORTANT: the dataset mirror contains PNG imagery/masks without tile georeferencing. The generated polygon GeoJSON files store image pixel coordinates (x=column, y=row), not latitude/longitude.\n\n"
    "Do not describe these pixel-coordinate files as georeferenced building footprints. For georeferenced output, run inference on GeoTIFFs with CRS and affine metadata.\n"
)
(ARTIFACT_DIR / 'README_attribution_and_limits.md').write_text(readme_text, encoding='utf-8')

print('Saved manifest and attribution note.')
print('Run folder:', RUN_DIR)
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(RUN_DIR), f'{path.stat().st_size/1024:.1f} KB')

## 16. Results interpretation and publication rules

- Use the **test split** only for the final report. Do not revisit thresholds or hyperparameters using test metrics.
- Report both micro (global-pixel) and macro (per-image average) Dice/IoU/Precision/Recall.
- Inspect visual examples and failure cases; one aggregate score does not describe boundary quality or building-instance counting.
- Do not claim the generated pixel-coordinate GeoJSON is geographically referenced.
- Do not claim performance from a run that stopped early because of runtime disconnection unless the saved history and checkpoint show a completed run.
- The downloadable WHU Hugging Face mirror declares CC BY 4.0. Preserve attribution and the original paper citation when publishing derived results.

### Resume wording after a valid completed run

Only after the complete run and test evaluation succeed, use measured values from `outputs/evaluation_summary.json`. A truthful project description can say that you designed and trained a TensorFlow U-Net for building-mask segmentation on the WHU Building Dataset, implemented paired image/mask loading, augmentation, tiled inference, validation-based threshold selection and evaluation. Do not write metrics until they are actually produced by the notebook.